# LoD2 Height Analysis

This notebook evaluates the coverage, quality, and distribution of the matched LoD2 building heights.

**Input**
- OSM buildings matched with LoD2 height information.

**Output**
- Summary statistics and quality assessment of the available height data.

In [1]:
# ============================================================
# 1 — SETUP: LOAD FINAL NATIONAL OSM + LoD2 DATASET
# ============================================================

from pathlib import Path
import gc
import numpy as np
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq

BASE_DIR=Path("/fast/home/o-olajuyigbe/data/germany_lod2")
FINAL_DIR=BASE_DIR/"final"
REPORT_DIR=BASE_DIR/"quality_reports"

OSM_LOD2_FILE=FINAL_DIR/"germany_buildings_classified_stage2_lod2.parquet"
MATCH_FILE=FINAL_DIR/"germany_osm_lod2_matches.parquet"

if not OSM_LOD2_FILE.exists(): raise FileNotFoundError(OSM_LOD2_FILE)
if not MATCH_FILE.exists(): raise FileNotFoundError(MATCH_FILE)


In [3]:

pf=pq.ParquetFile(OSM_LOD2_FILE)
columns=pf.schema_arrow.names

required=[
    "id","geometry","stage1_l1","stage1_l2",
    "lod2_height_raw_m","lod2_height_m","lod2_height_model_m",
    "lod2_match_quality","lod2_iou"
]

missing=[c for c in required if c not in columns]
if missing: raise KeyError(f"Missing required columns: {missing}")

print("="*72)
print("FINAL NATIONAL OSM + LoD2 DATASET")
print("="*72)
print(f"\nFile       : {OSM_LOD2_FILE}")
print(f"Rows       : {pf.metadata.num_rows:,}")
print(f"Row groups : {pf.metadata.num_row_groups:,}")
print(f"Columns    : {len(columns):,}")
print(f"Size       : {OSM_LOD2_FILE.stat().st_size/1024**3:.2f} GB")

print("\nLoD2 columns:")
for column in columns:
    if column.startswith("lod2_"):
        print(f"  {column}")

FINAL NATIONAL OSM + LoD2 DATASET

File       : /fast/home/o-olajuyigbe/data/germany_lod2/final/germany_buildings_classified_stage2_lod2.parquet
Rows       : 38,802,372
Row groups : 38
Columns    : 73
Size       : 5.83 GB

LoD2 columns:
  lod2_uid
  lod2_id
  lod2_state_code
  lod2_height_raw_m
  lod2_height_m
  lod2_height_model_m
  lod2_height_method
  lod2_footprint_method
  lod2_height_status
  lod2_height_available
  lod2_height_trusted
  lod2_height_nonpositive
  lod2_height_below_1m
  lod2_height_extreme
  lod2_match_quality
  lod2_match_pattern
  lod2_match_method
  lod2_iou
  lod2_osm_coverage
  lod2_footprint_coverage
  lod2_intersection_m2
  lod2_score
  lod2_score_margin
  lod2_reuse_count
  lod2_reused


In [4]:
# ============================================================
# 2 — LoD2 HEIGHT COVERAGE BY BUILDING CLASS
# ============================================================

cols=["stage1_l1","stage1_l2","lod2_height_raw_m","lod2_height_m","lod2_height_model_m","lod2_match_quality"]
df=pd.read_parquet(OSM_LOD2_FILE,columns=cols)

df["has_raw_height"]=df["lod2_height_raw_m"].notna()
df["has_plausible_height"]=df["lod2_height_m"].notna()
df["has_model_height"]=df["lod2_height_model_m"].notna()

def height_coverage(data,groups):
    out=(
        data.groupby(groups,dropna=False)
        .agg(
            buildings=("has_raw_height","size"),
            raw_heights=("has_raw_height","sum"),
            plausible_heights=("has_plausible_height","sum"),
            model_heights=("has_model_height","sum"),
            median_height_m=("lod2_height_model_m","median"),
            q25_height_m=("lod2_height_model_m",lambda x:x.quantile(0.25)),
            q75_height_m=("lod2_height_model_m",lambda x:x.quantile(0.75)),
            high_quality=("lod2_match_quality",lambda x:x.eq("high").sum()),
            medium_quality=("lod2_match_quality",lambda x:x.eq("medium").sum()),
            low_quality=("lod2_match_quality",lambda x:x.eq("low").sum())
        )
        .reset_index()
    )

    out["raw_coverage_pct"]=100*out["raw_heights"]/out["buildings"]
    out["plausible_coverage_pct"]=100*out["plausible_heights"]/out["buildings"]
    out["model_coverage_pct"]=100*out["model_heights"]/out["buildings"]
    out["low_quality_pct"]=100*out["low_quality"]/out["buildings"]
    return out

l1_height=height_coverage(df,["stage1_l1"])
l2_height=height_coverage(df,["stage1_l1","stage1_l2"])


print("="*80)
print("LoD2 HEIGHT COVERAGE BY L1 CLASS")
print("="*80)

display(
    l1_height.sort_values("buildings",ascending=False).round({
        "median_height_m":2,
        "q25_height_m":2,
        "q75_height_m":2,
        "raw_coverage_pct":2,
        "plausible_coverage_pct":2,
        "model_coverage_pct":2,
        "low_quality_pct":2
    })
)

print("\n"+"="*80)
print("LoD2 HEIGHT COVERAGE BY L2 CLASS")
print("="*80)

display(
    l2_height[l2_height["buildings"].ge(1000)]
    .sort_values(["stage1_l1","model_coverage_pct"],ascending=[True,False])
    .round({
        "median_height_m":2,
        "q25_height_m":2,
        "q75_height_m":2,
        "raw_coverage_pct":2,
        "plausible_coverage_pct":2,
        "model_coverage_pct":2,
        "low_quality_pct":2
    })
)

print(f"\nL1 report: {l1_height}")
print(f"L2 report: {l2_height}")

LoD2 HEIGHT COVERAGE BY L1 CLASS


,stage1_l1,buildings,raw_heights,plausible_heights,model_heights,median_height_m,q25_height_m,q75_height_m,high_quality,medium_quality,low_quality,raw_coverage_pct,plausible_coverage_pct,model_coverage_pct,low_quality_pct
9,NaN,25530280,22844755,22844718,21591483,7.84,4.52,9.75,11830787,9760733,1253235,89.48,89.48,84.57,4.91
6,residential,7765260,7482027,7482023,7210228,9.44,8.15,11.11,4202548,3007684,271795,96.35,96.35,92.85,3.50
3,filter,4121244,3335789,3335786,3135204,2.93,2.53,4.10,1963583,1171624,200582,80.94,80.94,76.07,4.87
2,commercial,424475,395441,395437,379926,8.13,5.49,11.60,254925,125005,15511,93.16,93.16,89.50,3.65
0,agricultural,313609,265454,265454,253315,6.76,4.80,8.96,154429,98886,12139,84.64,84.64,80.77,3.87
1,civic,311495,287796,287796,277149,10.16,7.08,14.63,183719,93430,10647,92.39,92.39,88.97,3.42
4,industrial,301728,246800,246798,233164,7.60,5.25,10.25,147933,85233,13634,81.80,81.79,77.28,4.52
8,transportation,22301,16925,16924,16291,8.23,5.17,12.06,10646,5646,633,75.89,75.89,73.05,2.84
5,military,8288,3694,3694,3502,6.00,3.97,11.62,2342,1160,192,44.57,44.57,42.25,2.32
7,semi_commercial,3692,3558,3558,3433,16.03,12.25,19.48,2597,836,125,96.37,96.37,92.98,3.39



LoD2 HEIGHT COVERAGE BY L2 CLASS


,stage1_l1,stage1_l2,buildings,raw_heights,plausible_heights,model_heights,median_height_m,q25_height_m,q75_height_m,high_quality,medium_quality,low_quality,raw_coverage_pct,plausible_coverage_pct,model_coverage_pct,low_quality_pct
2,agricultural,farm_auxiliary,241815,218026,218026,208326,7.11,5.26,9.21,124403,83923,9700,90.16,90.16,86.15,4.01
0,agricultural,animal_keeping,8553,7289,7289,6968,7.31,5.86,9.18,4475,2493,321,85.22,85.22,81.47,3.75
1,agricultural,equestrian,16263,13825,13825,13196,6.81,4.96,8.54,8139,5057,629,85.01,85.01,81.14,3.87
3,agricultural,greenhouse,46423,25819,25819,24359,3.91,3.10,4.90,17134,7225,1460,55.62,55.62,52.47,3.14
7,civic,community,16117,15645,15645,15202,8.56,6.22,11.21,9334,5868,443,97.07,97.07,94.32,2.75
6,civic,clinic,5929,5689,5689,5499,10.43,8.09,13.15,3375,2124,190,95.95,95.95,92.75,3.20
15,civic,religious,74603,70675,70675,68720,13.07,8.35,21.17,46788,21932,1955,94.73,94.73,92.11,2.62
10,civic,government_office,10583,10039,10039,9734,14.37,11.63,17.86,6694,3040,305,94.86,94.86,91.98,2.88
9,civic,emergency_service,25826,24617,24617,23701,7.83,6.11,9.94,13193,10508,916,95.32,95.32,91.77,3.55
5,civic,care_facility,9694,9193,9193,8848,12.43,9.33,15.56,6051,2797,345,94.83,94.83,91.27,3.56



L1 report:          stage1_l1  buildings  raw_heights  plausible_heights  model_heights  \
0     agricultural     313609       265454             265454         253315   
1            civic     311495       287796             287796         277149   
2       commercial     424475       395441             395437         379926   
3           filter    4121244      3335789            3335786        3135204   
4       industrial     301728       246800             246798         233164   
5         military       8288         3694               3694           3502   
6      residential    7765260      7482027            7482023        7210228   
7  semi_commercial       3692         3558               3558           3433   
8   transportation      22301        16925              16924          16291   
9              NaN   25530280     22844755           22844718       21591483   

   median_height_m  q25_height_m  q75_height_m  high_quality  medium_quality  \
0            6.760         